In [5]:
using Pkg
Pkg.activate("../18337")
using Gridap, Gridap.Geometry, Gridap.Fields, GridapGmsh
using LinearAlgebra

  Activating project at `c:\Rena\MIT2\18.337\18337`


In [6]:
model = GmshDiscreteModel("CantileverBeam.msh")

Info    : Reading 'CantileverBeam.msh'...
Info    : 13 entities
Info    : 7589 nodes
Info    : 15176 elements
Info    : Done reading 'CantileverBeam.msh'


UnstructuredDiscreteModel()

In [38]:
# Define FE space
order = 1
reffe = ReferenceFE(lagrangian, VectorValue{2,Float64}, order)

V = TestFESpace(
    model,
    reffe,
    conformity = :H1,
    dirichlet_tags = ["Clamp"]
)

U = TrialFESpace(V, VectorValue(0.0, 0.0))

TrialFESpace()

In [39]:
# Define integration over certain domains
degree = 2

Ω = Triangulation(model)
dΩ = Measure(Ω, degree)

Ω_d = Triangulation(model, tags = "Design")
dΩ_d = Measure(Ω_d, degree)

Γ_load = BoundaryTriangulation(model; tags = ["Load"])
dΓ_load = Measure(Γ_load, degree)

GenericMeasure()

In [40]:
# FE spaces for design parameters
p_reffe = ReferenceFE(lagrangian, Float64, 0)
Q = TestFESpace(Ω_d, p_reffe, vector_type = Vector{Float64})
P = Q
np = num_free_dofs(P) # Number of cells in design region (number of design parameters)

pf_reffe = ReferenceFE(lagrangian, Float64, 1)
Qf = TestFESpace(Ω_d, pf_reffe, vector_type = Vector{Float64})
Pf = Qf

fem_params = (; V, U, Q, P, Qf, Pf, np, Ω, dΩ, dΩ_d, dΓ_load)

(V = UnconstrainedFESpace(), U = TrialFESpace(), Q = UnconstrainedFESpace(), P = UnconstrainedFESpace(), Qf = UnconstrainedFESpace(), Pf = UnconstrainedFESpace(), np = 14776, Ω = BodyFittedTriangulation(), dΩ = GenericMeasure(), dΩ_d = GenericMeasure(), dΓ_load = GenericMeasure())

In [41]:
# PHYSICAL PARAMETERS
E0 = 2.5          # Young's modulus of solid material (GPa)
Emin = 1e-6       # small stiffness for "void" material
ν = 0.33          # Poisson's ratio
q_simp = 3        # SIMP penalization exponent
Vf = 0.4          # allowed volume fraction
L = 0.04          # beam length (m)
H = 0.01          # beam height (m)
t = 0.01          # thickness if using 2D plane stress as a physical plate/beam (m), not used if treating as a 2D plane strain problem
f_load = -1.0     # magnitude of downward load (N/m)
traction = VectorValue(0.0, f_load) # downward traction on right edge 
phys_params = (; E0, Emin, ν, q_simp, Vf, L, H, t, traction)

# PROBLEM PARAMETERS
r = 0.0005          # Helmholtz filter radius
β = 8                   # β∈[1,∞], threshold sharpness
η = 0.5                     # η∈[0,1], threshold center

0.5

In [42]:
a_f(r, u, v) = r^2 * (∇(v) ⋅ ∇(u))

function Filter(p0; r, fem_params) 
    ph = FEFunction(fem_params.P, p0)
    op = AffineFEOperator(fem_params.Pf, fem_params.Qf) do u, v
        ∫(a_f(r, u, v))fem_params.dΩ_d + ∫(v * u)fem_params.dΩ_d, ∫(v * ph)fem_params.dΩ_d
      end
    pfh = solve(op)
    return get_free_dof_values(pfh)
end

function Threshold(pfh; β, η)
    return ((tanh(β * η) + tanh(β * (pfh - η))) / (tanh(β * η) + tanh(β * (1.0 - η))))
end

Threshold (generic function with 1 method)

In [43]:
# Problem formulation
struct SIMP{PT} <: Function # New Julia function type for SIMP material model
    params::PT
end

function (s::SIMP)(ρ) # ρ is the design variable (density) at a point
    p = s.params.q_simp
    E0 = s.params.E0
    Emin = s.params.Emin
    return Emin + ρ^p * (E0 - Emin)
end

In [44]:
# Define elasticity tensors and stress-strain relationships
ε(u) = 0.5 * (∇(u) + ∇(u)')
λ(E, ν) = (E * ν) / ((1.0 + ν) * (1.0 - 2.0ν))
μ(E, ν) = E / (2.0 * (1.0 + ν))

const I2 = TensorValue(1.0, 0.0,
                       0.0, 1.0) # 2D identity tensor

function σ_from_E(u, E, poisson)
    lambda_val = λ(E, poisson)
    mu_val = μ(E, poisson)
    strain_u = ε(u)

    return lambda_val * tr(strain_u) * I2 +
           2.0 * mu_val * strain_u
end

function a_base_elastic(u, v; phys_params)
    Emin = phys_params.Emin
    ν = phys_params.ν

    return ε(v) ⊙ σ_from_E(u, Emin, ν)
end

function a_design_elastic(u, v, pth; phys_params)
    E0 = phys_params.E0
    Emin = phys_params.Emin
    ν = phys_params.ν
    q = phys_params.q_simp

    σ0 = σ_from_E(u, E0, ν)
    σmin = σ_from_E(u, Emin, ν)
    simp_pth = ((p -> p^q) ∘ pth)

    return (simp_pth) * (ε(v) ⊙ (σ0 - σmin))
end

function MatrixK(pth; phys_params, fem_params) # full stiffness matrix assembly through LU factorization
    K_mat = assemble_matrix(fem_params.U, fem_params.V) do u, v
        ∫(a_base_elastic(u, v; phys_params))fem_params.dΩ +
        ∫(a_design_elastic(u, v, pth; phys_params))fem_params.dΩ_d
    end

    return lu(K_mat)
end

MatrixK (generic function with 1 method)

In [45]:
# Initial forward solve
p0 = zeros(fem_params.np)  # Here we make p=0 everywhere just for illustration purpose
pf_vec = Filter(p0;r, fem_params)
pfh = FEFunction(fem_params.Pf, pf_vec)
pth = (pf -> Threshold(pf; β, η)) ∘ pfh
K_mat = MatrixK(pth; phys_params = phys_params, fem_params = fem_params)

F_vec = assemble_vector(fem_params.V) do v
    ∫(v ⋅ phys_params.traction)fem_params.dΓ_load
end

u_vec = K_mat \ F_vec
uh = FEFunction(fem_params.U, u_vec)
Γ_clamp = BoundaryTriangulation(model; tags=["Clamp"])
dΓ_clamp = Measure(Γ_clamp, degree)

clamp_motion = sum(∫(uh ⋅ uh)dΓ_clamp)

99.79321460555943

In [15]:
# Objective function (compliance)
function Compliance(u_vec, F_vec)
    return dot(F_vec, u_vec)
end

# Total volume of the design domain
function DesignVolume(; fem_params)
    return sum(∫(1.0)fem_params.dΩ_d)
end

# Physical volume fraction using thresholded density pth
function VolumeFraction(pth; fem_params)
    Vd = DesignVolume(fem_params=fem_params)
    return sum(∫(pth)fem_params.dΩ_d) / Vd
end

# Volume constraint: gV <= 0
function VolumeConstraint(pf_vec; β, η, phys_params, fem_params)
    pfh = FEFunction(fem_params.Pf, pf_vec)
    pth = (pf -> Threshold(pf; β=β, η=η)) ∘ pfh
    
    return VolumeFraction(pth; fem_params=fem_params) - phys_params.Vf
end

VolumeConstraint (generic function with 1 method)

### Optimization with Adjoint Method

In [16]:
using ChainRulesCore, Zygote
import ChainRulesCore: rrule
NO_FIELDS = ZeroTangent()

ZeroTangent()

In [17]:
# Some helper derivatives
Dptdpf(pf, β, η) = β * (1.0 - tanh(β * (pf - η))^2) / (tanh(β * η) + tanh(β * (1.0 - η)))

# Derivative of SIMP Young's Modulus w.r.t pf
function DEdpf(pf, phys_params, β, η)
    q = phys_params.q_simp
    E0 = phys_params.E0
    Emin = phys_params.Emin
    
    # Formula: dE/dpth * dpth/dpf
    dE_dpth = q * Threshold(pf; β=β, η=η)^(q - 1) * (E0 - Emin)
    return dE_dpth * Dptdpf(pf, β, η)
end

# Derivative of SIMP density factor w.r.t pf
function DSIMPdpf(pf, phys_params, β, η)
    q = phys_params.q_simp
    
    # Formula: d(pth^q)/dpth * dpth/dpf
    dSIMP_dpth = q * Threshold(pf; β=β, η=η)^(q - 1)
    return dSIMP_dpth * Dptdpf(pf, β, η)
end

# Derivative of the Elasticity Weak Form w.r.t pf
function DKdpf(u, v, pfh; phys_params, β, η)
    E0 = phys_params.E0
    Emin = phys_params.Emin
    ν = phys_params.ν
    
    # Base stress tensors
    σ0 = σ_from_E(u, E0, ν)
    σmin = σ_from_E(u, Emin, ν)
    
    # Map the SIMP derivative over the domain
    dSIMP_func = p -> DSIMPdpf(p, phys_params, β, η)
    
    return (dSIMP_func ∘ pfh) * (ε(v) ⊙ (σ0 - σmin))
end

# Compliance objective as a function of filtered density pf
function gf_pf(pf_vec; β, η, phys_params, fem_params)
    pfh = FEFunction(fem_params.Pf, pf_vec)
    pth = (pf -> Threshold(pf; β = β, η = η)) ∘ pfh
    
    # Solve forward displacement
    K_mat = MatrixK(pth; phys_params = phys_params, fem_params = fem_params)

    F_vec = assemble_vector(fem_params.V) do v
        ∫(v ⋅ phys_params.traction)fem_params.dΓ_load
    end

    u_vec = K_mat \ F_vec
    
    # Compliance objective: g = Fᵀu
    return dot(F_vec, u_vec)
end

#### Adjoint Objective Derivative (`Dgfdpf`)
# This calculates the gradient of the compliance with respect to the filtered variables pf.

function Dgfdpf(pf_vec; β, η, phys_params, fem_params)
    pfh = FEFunction(fem_params.Pf, pf_vec)
    pth = (pf -> Threshold(pf; β=β, η=η)) ∘ pfh
    
    # Solve forward displacement
    K_mat = MatrixK(pth; phys_params=phys_params, fem_params=fem_params)
    F_vec = assemble_vector(v -> ∫(v ⋅ phys_params.traction)fem_params.dΓ_load, fem_params.V)
    u_vec = K_mat \ F_vec
    uh = FEFunction(fem_params.U, u_vec)
    
    # Compliance is self-adjoint: adjoint field w = uh
    # Sensitivity = -∫ [ uh * (dK/dpf) * uh ] dΩ
    l_temp(dp) = ∫( -1.0 * DKdpf(uh, uh, pfh; phys_params=phys_params, β=β, η=η) * dp )fem_params.dΩ_d
    
    return assemble_vector(l_temp, fem_params.Qf)
end


#### Filter Adjoint (`Dgdp`)
# This propagates the gradient from the filtered space pf back to the design variables p.

function Dgdp(dgdpf; r, fem_params)
    # Assemble the Filter PDE matrix (Af)
    Af = assemble_matrix(fem_params.Pf, fem_params.Qf) do u, v
        ∫(a_f(r, u, v))fem_params.dΩ_d + ∫(v * u)fem_params.dΩ_d
    end
    
    # Adjoint solve for the filter (Af is symmetric)
    wvec = Af' \ dgdpf
    wh = FEFunction(fem_params.Pf, wvec)
    
    # Convert sensitivity back to the piece-wise constant space P
    return assemble_vector(dp -> ∫(wh * dp)fem_params.dΩ_d, fem_params.P)
end

Dgdp (generic function with 1 method)

In [18]:
### Custom adjoint rules
# Helmholtz filter forward and backward rules for AD
# Forward map: p -> pf
function pf_p0(p0; r, fem_params)
    pf_vec = Filter(p0; r = r, fem_params = fem_params)
    return pf_vec
end

# Custom reverse rule for p -> pf
function ChainRulesCore.rrule(::typeof(pf_p0), p0; r, fem_params)

    function pf_pullback(dgdpf)
        # dgdpf is ∂g/∂pf coming from later in the computational graph
        dgdpf_vec = unthunk(dgdpf)

        # Pull gradient back through the Helmholtz filter:
        # dg/dp = Mᵀ Af^{-T} dg/dpf
        dgdp = Dgdp(dgdpf_vec; r = r, fem_params = fem_params)

        return NoTangent(), dgdp
    end

    return pf_p0(p0; r = r, fem_params = fem_params), pf_pullback
end

# Custom reverse rule for pf -> compliance
function ChainRulesCore.rrule(::typeof(gf_pf), pf_vec; β, η, phys_params, fem_params)

    function gf_pullback(dgdg)
        # dgdg is the upstream scalar sensitivity.
        # Usually dgdg = 1.0 if this is the final scalar objective.
        dgdg_val = unthunk(dgdg)

        # Local derivative: dg/dpf
        dgfdpf = Dgfdpf(pf_vec; β = β, η = η, phys_params = phys_params, fem_params = fem_params)

        return NoTangent(), dgdg_val * dgfdpf
    end

    return gf_pf(pf_vec; β = β, η = η, phys_params = phys_params, fem_params = fem_params), gf_pullback
end

In [19]:
# FULL compliance objective as a function of raw design variables p0
function gf_p(p0::Vector; r, β, η, phys_params, fem_params)
    pf_vec = pf_p0(p0; r = r, fem_params = fem_params)
    gf_pf(pf_vec; β = β, η = η, phys_params = phys_params, fem_params = fem_params)
end

# compute dg/dp_0 using Zygote and the custom adjoint rules defined above
# Calculated gradient AND objective value for NLopt
# NLopt-compatible compliance objective
function gf_p(p0::Vector, grad::Vector; r, β, η, phys_params, fem_params)
    if length(grad) > 0
        dgdp, = Zygote.gradient(p -> gf_p(p; r=r, β=β, η=η, phys_params=phys_params, fem_params=fem_params), p0)
        grad[:] = dgdp
    end

    gvalue = gf_p(p0; r=r, β=β, η=η, phys_params=phys_params, fem_params=fem_params)

    open("beam_mma/gvalue.txt", "a") do io
        write(io, "$gvalue \n")
    end

    return gvalue
end

gf_p (generic function with 2 methods)

In [20]:
# using Printf
# # Test the gradient implementation by comparing to finite differences
# βtest = 8.0
# ηtest = 0.5

# p0 = rand(fem_params.np)

# d = randn(fem_params.np)
# d ./= norm(d)

# grad = zeros(fem_params.np)
# g0 = gf_p(p0, grad; r=r, β=βtest, η=ηtest, phys_params=phys_params, fem_params=fem_params)

# adj_dir = dot(grad, d)

# println("g0 = ", g0)
# println("norm grad = ", norm(grad))
# println("grad extrema = ", extrema(grad))
# println("adj_dir = ", adj_dir)

# for h in [1e-2, 1e-3, 1e-4, 1e-5, 1e-6]
#     g1 = gf_p(p0 + h*d, []; r=r, β=βtest, η=ηtest, phys_params=phys_params, fem_params=fem_params)

#     fd_dir = (g1 - g0) / h
#     rel_error = abs(fd_dir - adj_dir) / max(abs(fd_dir), abs(adj_dir), eps())

#     @printf("h = %.1e, fd_dir = %.8e, adj_dir = %.8e, rel_error = %.8e\n",
#             h, fd_dir, adj_dir, rel_error)
# end

Negative extrema: increasing density decreases compliance (as expected)

Moving in random direction d decreases compliance locally

Error improves as h decreases, then gets worse again as roundoff dominates

In [21]:
dgdpf = Dgfdpf(pf_vec; β=8.0, η=0.5, phys_params=phys_params, fem_params=fem_params)
println("norm dgdpf = ", norm(dgdpf), ", extrema = ", extrema(dgdpf))

dgdp = Dgdp(dgdpf; r=r, fem_params=fem_params)
println("norm dgdp = ", norm(dgdp), ", extrema = ", extrema(dgdp))

grad = zeros(fem_params.np)
g0 = gf_p(p0, grad; r=r, β=8.0, η=0.5, phys_params=phys_params, fem_params=fem_params)
println("norm grad from gf_p = ", norm(grad), ", extrema = ", extrema(grad))

norm dgdpf = 0.0, extrema = (0.0, 0.0)
norm dgdp = 0.0, extrema = (0.0, 0.0)
norm grad from gf_p = 0.0, extrema = (0.0, 0.0)


In [22]:
# Derivative of volume constraint w.r.t filtered variables pf
function DgVdpf(pf_vec; β, η, phys_params, fem_params)
    pfh = FEFunction(fem_params.Pf, pf_vec)
    Vd = DesignVolume(fem_params=fem_params)
    
    # Map threshold derivative over the domain
    dpt_func = pf -> Dptdpf(pf, β, η)
    
    # Formula: dVolume/dpf = (1 / |Ωd|) * dpt/dpf
    l_temp(dp) = ∫( dp * (dpt_func ∘ pfh) / Vd )fem_params.dΩ_d
    
    return assemble_vector(l_temp, fem_params.Qf)
end

# Derivative of volume constraint w.r.t raw design variables p
function DgVdp(p0; r, β, η, phys_params, fem_params)
    pf_vec = pf_p0(p0; r=r, fem_params=fem_params)
    
    # Compute derivative w.r.t filtered variables pf
    dgVdpf = DgVdpf(pf_vec; β=β, η=η, phys_params=phys_params, fem_params=fem_params)
    
    # Pull gradient back through the Helmholtz filter
    return Dgdp(dgVdpf; r=r, fem_params=fem_params)
end

# NLopt-compatible volume constraint
function gV_p(p0::Vector, grad::Vector; r, β, η, phys_params, fem_params)
    pf_vec = pf_p0(p0; r=r, fem_params=fem_params)
    
    if length(grad) > 0
        grad[:] = DgVdp(p0; r=r, β=β, η=η, phys_params=phys_params, fem_params=fem_params)
    end
    
    gVvalue = VolumeConstraint(pf_vec; β=β, η=η, phys_params=phys_params, fem_params=fem_params)
    
    return gVvalue
end
# if gV_p(p0, grad; r=r, β=βtest, η=ηtest, phys_params=phys_params, fem_params=fem_params) <= 0, constraint is satisfied

gV_p (generic function with 1 method)

In [ ]:
# # Test the volume constraint gradient implementation by comparing to finite differences
# βtest = 1.0
# ηtest = 0.5

# p0 = rand(fem_params.np)

# d = randn(fem_params.np)
# d ./= norm(d)

# gradV = zeros(fem_params.np)
# gV0 = gV_p(p0, gradV; r=r, β=βtest, η=ηtest, phys_params=phys_params, fem_params=fem_params)

# adj_dir = dot(gradV, d)

# println("gV0 = ", gV0)
# println("norm gradV = ", norm(gradV))
# println("gradV extrema = ", extrema(gradV))
# println("adj_dir = ", adj_dir)

# for h in [1e-2, 1e-3, 1e-4, 1e-5, 1e-6]
#     gV1 = gV_p(p0 + h*d, []; r=r, β=βtest, η=ηtest, phys_params=phys_params, fem_params=fem_params)

#     fd_dir = (gV1 - gV0) / h
#     rel_error = abs(fd_dir - adj_dir) / max(abs(fd_dir), abs(adj_dir), eps())

#     @printf("h = %.1e, fd_dir = %.8e, adj_dir = %.8e, rel_error = %.8e\n",
#             h, fd_dir, adj_dir, rel_error)
# end

gV0 = 0.09666806081332768
norm gradV = 0.008900274806827466
gradV extrema = (4.317071718070394e-5, 0.00010292839959508143)
adj_dir = -4.6034665976246054e-5


LoadError: LoadError: UndefVarError: `@printf` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
Hint: a global variable of this name also exists in Printf.
in expression starting at In[23]:26

### Optimization with MMA

In [24]:
# Doubling beta scheme (Sigmund 2007)
# Minimize objective
# Include volume constraint
# Runs a sequence of MMA optimizations, each for 50 iterations, and feeds the previous optimized design into the next sharper projection stage.
using NLopt
using DelimitedFiles
using Serialization

# NLopt for fixed beta 
function gf_p_optimize(p_init; r, β, η, TOL = 1e-4, MAX_ITER = 50, phys_params, fem_params, history = nothing)
    opt = Opt(:LD_MMA, fem_params.np)

    opt.lower_bounds = zeros(fem_params.np)
    opt.upper_bounds = ones(fem_params.np)

    opt.ftol_rel = TOL
    opt.maxeval = MAX_ITER

    # Minimize compliance: g(p) = Fᵀu
    opt.min_objective = (p0, grad) -> begin
        gvalue = gf_p(p0, grad; r=r, β=β, η=η, phys_params=phys_params, fem_params=fem_params)

        if history !== nothing
            push!(history[:g], gvalue)
            push!(history[:β], β)
            push!(history[:η], η)
            push!(history[:vol], gV_p(p0, []; r=r, β=β, η=η, phys_params=phys_params, fem_params=fem_params) + phys_params.Vf)
        end

        return gvalue
    end

    # Volume constraint: gV(p) <= 0
    inequality_constraint!(opt, (p0, grad) -> gV_p(p0, grad; r=r, β=β, η=η, phys_params=phys_params, fem_params=fem_params), 1e-8)

    (g_opt, p_opt, ret) = optimize(opt, p_init)

    @show numevals = opt.numevals

    return g_opt, p_opt, ret
end

# Continuation scheme with doubling beta
function gf_p_optimize_doubling(p_init; r, η, TOL = 1e-8, ITER_PER_BETA = 50, β_START = 1.0, β_MAX = 512.0, phys_params, fem_params, save_prefix = "beam_mma")
    ##################### Initialize #####################
    p_opt = copy(p_init)
    g_opt = Inf
    β = β_START

    history = Dict(
        :g => Float64[],
        :β => Float64[],
        :η => Float64[],
        :vol => Float64[],
        :stage_g => Float64[],
        :stage_β => Float64[],
        :stage_η => Float64[],
        :stage_vol => Float64[],
        :stage_ret => String[]
    )

    if !isdir(save_prefix)
        mkdir(save_prefix)
    end

    stage = 1

    ##################### Continuation #####################
    while β <= β_MAX
        println("Starting MMA stage $stage with β = $β, η = $η")

        g_opt, p_temp_opt, ret = gf_p_optimize(
            p_opt;
            r = r,
            β = β,
            η = η,
            TOL = TOL,
            MAX_ITER = ITER_PER_BETA,
            phys_params = phys_params,
            fem_params = fem_params,
            history = history
        )

        p_opt = copy(p_temp_opt)

        vol = gV_p(p_opt, []; r=r, β=β, η=η, phys_params=phys_params, fem_params=fem_params) + phys_params.Vf

        push!(history[:stage_g], g_opt)
        push!(history[:stage_β], β)
        push!(history[:stage_η], η)
        push!(history[:stage_vol], vol)
        push!(history[:stage_ret], string(ret))

        # Save checkpoint p and metadata
        serialize(joinpath(save_prefix, "p_stage_$(stage)_beta_$(Int(round(β))).jls"), p_opt)

        open(joinpath(save_prefix, "stage_history.csv"), "a") do io
            write(io, "$stage,$β,$η,$g_opt,$vol,$ret\n")
        end

        global_stage_msg = "Finished stage $stage: β = $β, g = $g_opt, volume = $vol, ret = $ret"
        println(global_stage_msg)

        β *= 2.0
        stage += 1
    end

    ##################### Save full history #####################
    writedlm(joinpath(save_prefix, "g_history.csv"), hcat(history[:g], history[:β], history[:η], history[:vol]), ',')

    serialize(joinpath(save_prefix, "p_final.jls"), p_opt)

    return g_opt, p_opt, history
end

gf_p_optimize_doubling (generic function with 1 method)

### Comparison: Finite Difference

In [26]:
using Printf

In [27]:
# Bounded finite difference gradient for scalar function f(p)
function finite_difference_gradient(f, p; h=1e-6, lower=0.0, upper=1.0)
    grad = zeros(length(p))
    p_plus = copy(p)
    p_minus = copy(p)

    f0 = f(p)
    for i in eachindex(p)
        if i % 50 == 0
            println("Computing finite difference for parameter index $i")
        end
        pi = p[i]

        # Reset perturbation vectors
        p_plus[i] = pi
        p_minus[i] = pi

        if pi - h >= lower && pi + h <= upper
            # Central difference:
            # ∂f/∂p_i ≈ [f(p + h e_i) - f(p - h e_i)] / (2h)
            p_plus[i] = pi + h
            p_minus[i] = pi - h

            f_plus = f(p_plus)
            f_minus = f(p_minus)

            grad[i] = (f_plus - f_minus) / (2.0h)

        elseif pi + h <= upper
            # Forward difference near lower bound:
            # ∂f/∂p_i ≈ [f(p + h e_i) - f(p)] / h
            p_plus[i] = pi + h

            f_plus = f(p_plus)

            grad[i] = (f_plus - f0) / h

        elseif pi - h >= lower
            # Backward difference near upper bound:
            # ∂f/∂p_i ≈ [f(p) - f(p - h e_i)] / h
            p_minus[i] = pi - h

            f_minus = f(p_minus)

            grad[i] = (f0 - f_minus) / h

        else
            grad[i] = 0.0
        end

        # Restore entries
        p_plus[i] = pi
        p_minus[i] = pi
    end

    return grad
end

finite_difference_gradient (generic function with 1 method)

In [28]:
# Finite difference gradient of compliance objective
function FD_dgdp(p0; r, β, η, phys_params, fem_params, h=1e-6)
    # Objective as scalar function of raw design variables p
    f_obj = p -> gf_p(p; r=r, β=β, η=η, phys_params=phys_params, fem_params=fem_params)

    return finite_difference_gradient(f_obj, p0; h=h, lower=0.0, upper=1.0)
end

# NLopt-compatible compliance objective using finite difference gradients
function gf_p_fd(p0::Vector, grad::Vector; r, β, η, phys_params, fem_params, h_fd=1e-6)
    if length(grad) > 0
        
        grad[:] = FD_dgdp(p0; r=r, β=β, η=η, phys_params=phys_params, fem_params=fem_params, h=h_fd)
    end

    # Same objective value as the adjoint version
    gvalue = gf_p(p0; r=r, β=β, η=η, phys_params=phys_params, fem_params=fem_params)

    return gvalue
end

gf_p_fd (generic function with 1 method)

In [ ]:
# Same as gf_p_optimize but using finite difference gradients instead of adjoint gradients
# NLopt for fixed beta 
function gf_p_optimize_fd(p_init; r, β, η, TOL=1e-4, MAX_ITER=50, h_fd=1e-6, phys_params, fem_params, history=nothing)
    opt = Opt(:LD_MMA, fem_params.np)

    opt.lower_bounds = zeros(fem_params.np)
    opt.upper_bounds = ones(fem_params.np)

    opt.ftol_rel = TOL
    opt.maxeval = MAX_ITER

    # Minimize compliance: g(p) = Fᵀu
    opt.min_objective = (p0, grad) -> begin
        gvalue = gf_p_fd(p0, grad; r=r, β=β, η=η, phys_params=phys_params, fem_params=fem_params, h_fd=h_fd)

        if history !== nothing
            push!(history[:g], gvalue)
            push!(history[:β], β)
            push!(history[:η], η)
            push!(history[:vol], gV_p(p0, []; r=r, β=β, η=η, phys_params=phys_params, fem_params=fem_params) + phys_params.Vf)
        end

        return gvalue
    end

    # Volume constraint: gV(p) <= 0
    inequality_constraint!(
        opt,
        (p0, grad) -> gV_p(p0, grad; r=r, β=β, η=η, phys_params=phys_params, fem_params=fem_params), # Use adjoint gradient for speed
        1e-8
    )

    (g_opt, p_opt, ret) = optimize(opt, p_init)

    @show numevals = opt.numevals

    return g_opt, p_opt, ret
end

gf_p_optimize_fd (generic function with 1 method)

In [30]:
# Continuation scheme with doubling beta using finite difference gradients
# Reduced iter per beta and beta max to keep runtime reasonable for testing
function gf_p_optimize_doubling_fd(p_init; r, η, TOL=1e-4, ITER_PER_BETA=10, β_START=1.0, β_MAX=8.0, h_fd=1e-6, phys_params, fem_params, save_prefix="beam_mma_fd")
    ##################### Initialize #####################
    p_opt = copy(p_init)
    g_opt = Inf
    β = β_START

    history = Dict(
        :g => Float64[],
        :β => Float64[],
        :η => Float64[],
        :vol => Float64[],
        :stage_g => Float64[],
        :stage_β => Float64[],
        :stage_η => Float64[],
        :stage_vol => Float64[],
        :stage_ret => String[]
    )

    if !isdir(save_prefix)
        mkdir(save_prefix)
    end

    stage = 1

    ##################### Continuation #####################
    while β <= β_MAX
        println("Starting FD-MMA stage $stage with β = $β, η = $η")

        g_opt, p_temp_opt, ret = gf_p_optimize_fd(
            p_opt;
            r=r,
            β=β,
            η=η,
            TOL=TOL,
            MAX_ITER=ITER_PER_BETA,
            h_fd=h_fd,
            phys_params=phys_params,
            fem_params=fem_params,
            history=history
        )

        p_opt = copy(p_temp_opt)

        vol = gV_p(p_opt, []; r=r, β=β, η=η, phys_params=phys_params, fem_params=fem_params) + phys_params.Vf

        push!(history[:stage_g], g_opt)
        push!(history[:stage_β], β)
        push!(history[:stage_η], η)
        push!(history[:stage_vol], vol)
        push!(history[:stage_ret], string(ret))

        println("Finished FD stage $stage: β = $β, g = $g_opt, volume = $vol, ret = $ret")

        β *= 2.0
        stage += 1
    end

    return g_opt, p_opt, history
end

gf_p_optimize_doubling_fd (generic function with 1 method)

In [31]:
# ACCURACY TEST: Compare finite difference gradients to adjoint gradients for the same design point, get cosine similarity and relative error, and report timings for both methods
function compare_gradient_accuracy(p0; r, β, η, phys_params, fem_params, h_fd=1e-6)
    println("Entered compare_gradient_accuracy")
    println("np = ", fem_params.np)
    println("length(p0) = ", length(p0))
    flush(stdout)
    ##################### Adjoint Gradient #####################
    grad_adj = zeros(fem_params.np)
    println("About to call gf_p...")
    flush(stdout)
    t_adj = @elapsed begin
        g_adj = gf_p(p0, grad_adj; r=r, β=β, η=η, phys_params=phys_params, fem_params=fem_params)
    end
    println("Adjoint gradient norm = ", norm(grad_adj))

    ##################### Finite Difference Gradient #####################
    println("Started FD computation...")
    t_fd = @elapsed begin
        grad_fd = FD_dgdp(p0; r=r, β=β, η=η, phys_params=phys_params, fem_params=fem_params, h=h_fd)
    end

    g_val = gf_p(p0; r=r, β=β, η=η, phys_params=phys_params, fem_params=fem_params)

    ##################### Accuracy Metrics #####################
    rel_error = norm(grad_adj - grad_fd) / max(norm(grad_adj), norm(grad_fd), eps())
    abs_error = norm(grad_adj - grad_fd)
    cosine_similarity = dot(grad_adj, grad_fd) / (norm(grad_adj) * norm(grad_fd) + eps())

    println("Objective g = ", g_val)
    println("Adjoint gradient time = ", t_adj)
    println("Finite difference gradient time = ", t_fd)
    println("Speedup FD / adjoint = ", t_fd / t_adj)
    println("Relative gradient error = ", rel_error)
    println("Absolute gradient error = ", abs_error)
    println("Cosine similarity = ", cosine_similarity)
    println("norm grad_adj = ", norm(grad_adj))
    println("norm grad_fd  = ", norm(grad_fd))

    return (
        g = g_val,
        grad_adj = grad_adj,
        grad_fd = grad_fd,
        t_adj = t_adj,
        t_fd = t_fd,
        rel_error = rel_error,
        abs_error = abs_error,
        cosine_similarity = cosine_similarity
    )
end


compare_gradient_accuracy (generic function with 1 method)

In [46]:
MAX_ITER = 1
p_init = fill(phys_params.Vf, fem_params.np)
println("Running adjoint MMA...")
println("np = ", fem_params.np)
println("MAX_ITER = ", MAX_ITER)
println("β = ", β)
println("η = ", η)
println("p_init extrema = ", extrema(p_init))


t_adj = @elapsed begin
    println("About to enter gf_p_optimize")
    flush(stdout)

    g_adj, p_adj, ret_adj = gf_p_optimize(
        p_init;
        r = r,
        β = β,
        η = η,
        TOL = 1e-6,
        MAX_ITER = MAX_ITER,
        phys_params = phys_params,
        fem_params = fem_params,
        history = nothing
    )

    println("Finished gf_p_optimize")
end

println("Adjoint MMA finished")
println("t_adj = ", t_adj)
println("g_adj = ", g_adj)
println("ret_adj = ", ret_adj)

Running adjoint MMA...
np = 14776
MAX_ITER = 1
β = 8
η = 0.5
p_init extrema = (0.4, 0.4)
About to enter gf_p_optimize
numevals = opt.numevals = 1
Finished gf_p_optimize
Adjoint MMA finished
t_adj = 38.1619728
g_adj = 0.00509645855468608
ret_adj = MAXEVAL_REACHED


In [34]:
vol_adj = gV_p(p_adj, []; r=r, β=β, η=η, phys_params=phys_params, fem_params=fem_params) + phys_params.Vf

0.1677587805935318

In [ ]:
# DO NOT RUN
print(

In [35]:
println("Running finite-difference MMA...")
h_fd = 1e-6
t_fd = @elapsed begin
    g_fd, p_fd, ret_fd = gf_p_optimize_fd(p_init; r=r, β=β, η=η, TOL=1e-6, MAX_ITER=1, h_fd=h_fd, phys_params=phys_params, fem_params=fem_params, history=nothing)
end


Running finite-difference MMA...
Computing finite difference for parameter index 50
Computing finite difference for parameter index 100
Computing finite difference for parameter index 150
Computing finite difference for parameter index 200
Computing finite difference for parameter index 250
Computing finite difference for parameter index 300
Computing finite difference for parameter index 350
Computing finite difference for parameter index 400
Computing finite difference for parameter index 450
Computing finite difference for parameter index 500
Computing finite difference for parameter index 550
Computing finite difference for parameter index 600
Computing finite difference for parameter index 650
Computing finite difference for parameter index 700
Computing finite difference for parameter index 750
Computing finite difference for parameter index 800
Computing finite difference for parameter index 850
Computing finite difference for parameter index 900
Computing finite difference for 

LoadError: MethodError: no method matching gV_p(::Vector{…}, ::Vector{…}; r::Float64, β::Int64, η::Float64, phys_params::@NamedTuple{…}, fem_params::@NamedTuple{…}, h_fd::Float64)
This method does not support all of the given keyword arguments (and may not support any).

[0mClosest candidates are:
[0m  gV_p(::Vector, ::Vector; r, β, η, phys_params, fem_params)[91m got unsupported keyword argument "h_fd"[39m
[0m[90m   @[39m [32mMain[39m [90m[4mIn[22]:27[24m[39m

Stacktrace:
  [1] [0m[1mkwerr[22m[0m[1m([22m::[0m@NamedTuple[90m{…}[39m, ::[0mFunction, ::[0mVector[90m{…}[39m, ::[0mVector[90m{…}[39m[0m[1m)[22m
[90m    @[39m [90mBase[39m [90m.\[39m[90m[4merror.jl:175[24m[39m
  [2] [0m[1m(::var"#103#104"{…})[22m[0m[1m([22m[90mp0[39m::[0mVector[90m{…}[39m, [90mgrad[39m::[0mVector[90m{…}[39m[0m[1m)[22m
[90m    @[39m [32mMain[39m [90m.\[39m[90m[4mIn[29]:29[24m[39m
  [3] [0m[1mnlopt_callback_wrapper[22m[0m[1m([22m[90mn[

^Literally also uses the adjoint version of volume gradient, so not a full calculation at all

Wall clock time: 171m52.9s

In [37]:
Γ_clamp = BoundaryTriangulation(model; tags=["Clamp"])
dΓ_clamp = Measure(Γ_clamp, degree)

clamp_motion = sum(∫(uh ⋅ uh)dΓ_clamp)

println("clamp displacement norm squared = ", clamp_motion)

clamp displacement norm squared = 99.79321460555943


In [ ]:
vol_fd = gV_p(p_fd, []; r=r, β=β, η=η, phys_params=phys_params, fem_params=fem_params) + phys_params.Vf
println("========== Optimization Runtime Comparison ==========")
println("Adjoint:")
println("  g      = ", g_adj)
#println("  vol    = ", vol_adj)
println("  ret    = ", ret_adj)
println("  time   = ", t_adj)

println("Finite Difference:")
println("  g      = ", g_fd)
println("  vol    = ", vol_fd)
println("  ret    = ", ret_fd)
println("  time   = ", t_fd)

println("FD / adjoint runtime ratio = ", t_fd / t_adj)